# BDC Satria Data 2026 — 6 Model Belum Ada di Lane Dosen (1 Notebook, Summary + Excel di Akhir)

Model yang dilatih di notebook ini (semua dikonfirmasi TIDAK ada di 19 lane dosen kamu):
1. **DeiT-Base** — `deit_base_patch16_224.fb_in1k`, ~86.6M params
2. **DINOv2 ViT-Base** — `vit_base_patch14_dinov2.lvd142m`, ~86M params (dosen cuma punya versi Small)
3. **RegNetY-16GF** — torchvision `regnet_y_16gf`, ~83M params
4. **DenseNet-264** — `densenet264d`, ~73M params
5. **CoAtNet-2** — `coatnet_2_rw_224.sw_in12k`, ~75M params (hybrid conv+attention)
6. **PVTv2-B5** — `pvt_v2_b5.in1k`, ~82M params

**Semua di resolusi 224** — konsisten dengan temuan kamu sendiri bahwa resolusi tinggi tidak selalu membantu (kadang malah lambat tanpa gain, seperti kasus EfficientNetV2-L kemarin), dan supaya waktu training 6 model total masih realistis.

**Total parameter yang benar-benar DILATIH (bukan total model):** karena semua pakai backbone-freeze + head-only, cuma head classifier kecil yang dilatih di tiap model — estimasi ~25.000-30.000 parameter gabungan ke-6 model (bukan ratusan juta). Angka pasti tiap model akan tercetak otomatis di fold pertama masing-masing.

**PERINGATAN — 2 model berisiko nama checkpoint kurang lazim:** `densenet264d` dan `coatnet_2_rw_224.sw_in12k` kemungkinan kecil pretrained checkpoint-nya tidak persis sesuai dugaan. Kode ini pakai **try/except per model** — kalau 1 model gagal load, notebook TETAP LANJUT ke 5 model lainnya (dicatat di ringkasan sebagai gagal, bukan bikin seluruh notebook crash).

**Konsisten dgn notebook lain:** `StratifiedShuffleSplit` 3-fold 90/10, class weight balanced per fold, macro F1, tracking misclassified digabung ke `.xlsx` 2 sheet (kolom `model` ditambah biar bisa dibedain).

## 1. Setup & Konfigurasi

In [1]:
from __future__ import annotations
import os
import gc
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torchvision import models as tv_models, transforms as T
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from PIL import Image

try:
    import timm
except ImportError:
    raise ImportError("timm belum terinstall. Jalankan: pip install timm")

c:\anaconda3\envs\malaria\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# === PATH DATASET ===
DATA_ROOT = Path(r"C:\Users\MyPC PRO\Downloads\BDC2026")
TRAIN_DIR = DATA_ROOT / "train"
MODELS_DIR = DATA_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = DATA_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

CLASS_FOLDERS = ["0_Recyclable", "1_Electronic", "2_Organic"]
CLASS_NAMES = ["Recyclable", "Electronic", "Organic"]
NUM_CLASSES = len(CLASS_NAMES)

# === MODEL LIST ===
MODEL_LIST = [
    {"key": "deit_base",       "type": "timm", "timm_name": "deit_base_patch16_224.fb_in1k",         "img_size": 224},
    {"key": "dinov2_base",     "type": "timm", "timm_name": "vit_base_patch14_dinov2.lvd142m",        "img_size": 224},
    {"key": "regnety_16gf",    "type": "torchvision_regnet",                                          "img_size": 224},
    {"key": "densenet264",     "type": "timm", "timm_name": "densenet264d",                            "img_size": 224},
    {"key": "coatnet_2",       "type": "timm", "timm_name": "coatnet_2_rw_224.sw_in12k",               "img_size": 224},
    {"key": "pvtv2_b5",        "type": "timm", "timm_name": "pvt_v2_b5.in1k",                          "img_size": 224},
]

# === CONFIG TRAINING (RTX 3060 12.9GB) ===
BATCH_SIZE = 128          # seragam utk semua 6 model (semua kelas ~75-90M @ res 224)
GRAD_ACCUM_STEPS = 1
NUM_EPOCHS = 15
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 4
N_FOLDS = 3
FOLD_VAL_FRAC = 0.10
SEED = 42
NUM_WORKERS = 8

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Jumlah model dalam antrian: {len(MODEL_LIST)}")
for m in MODEL_LIST:
    print(f"  - {m['key']}")

Device: cuda
GPU: NVIDIA GeForce RTX 3060
VRAM: 12.9 GB
Jumlah model dalam antrian: 6
  - deit_base
  - dinov2_base
  - regnety_16gf
  - densenet264
  - coatnet_2
  - pvtv2_b5


## 2. Index Dataset dari Folder

In [3]:
def index_dataset(train_dir: Path, class_folders: list[str]):
    records = []
    for label_idx, folder_name in enumerate(class_folders):
        folder = train_dir / folder_name
        if not folder.is_dir():
            raise FileNotFoundError(f"Folder tidak ditemukan: {folder}")
        exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        files = [p for p in folder.iterdir() if p.suffix.lower() in exts]
        for p in files:
            records.append({"path": str(p), "label": label_idx, "class_name": CLASS_NAMES[label_idx]})
    return pd.DataFrame(records)

full_df = index_dataset(TRAIN_DIR, CLASS_FOLDERS)
print(f"Total citra ter-index: {len(full_df)}")
print(full_df["class_name"].value_counts())

Total citra ter-index: 26527
class_name
Organic       12567
Recyclable     9999
Electronic     3961
Name: count, dtype: int64


## 3. Dataset & Transform

In [4]:
# TrashDataset & build_transforms dipindah ke dataset_utils_six.py (wajib jadi modul .py asli
# biar DataLoader num_workers>0 gak hang di Windows/Jupyter - spawn worker perlu import modul nyata)
from dataset_utils_six import TrashDataset, build_transforms

## 4. Model Builder (generik: timm ATAU torchvision RegNet)

In [5]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def build_model(model_cfg: dict, num_classes: int):
    if model_cfg["type"] == "timm":
        model = timm.create_model(model_cfg["timm_name"], pretrained=True, num_classes=num_classes)
        for p in model.parameters():
            p.requires_grad = False
        head_module = None
        for attr in ["head", "classifier", "fc"]:
            if hasattr(model, attr):
                candidate = getattr(model, attr)
                if isinstance(candidate, nn.Module):
                    head_module = candidate
                    break
        if head_module is None:
            raise AttributeError(f"Head classifier tidak ketemu untuk {model_cfg['timm_name']}")
        for p in head_module.parameters():
            p.requires_grad = True
        data_cfg = timm.data.resolve_model_data_config(model)
        mean, std = data_cfg["mean"], data_cfg["std"]

    elif model_cfg["type"] == "torchvision_regnet":
        model = tv_models.regnet_y_16gf(weights=tv_models.RegNet_Y_16GF_Weights.IMAGENET1K_V2)
        for p in model.parameters():
            p.requires_grad = False
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
        for p in model.fc.parameters():
            p.requires_grad = True
        mean, std = IMAGENET_MEAN, IMAGENET_STD

    else:
        raise ValueError(f"Tipe model tidak dikenal: {model_cfg['type']}")

    return model.to(DEVICE), mean, std

## 5. Train & Evaluate

In [6]:
scaler = GradScaler("cuda", enabled=(DEVICE == "cuda"))


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    optimizer.zero_grad()

    for step, (imgs, labels, _) in enumerate(loader):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast("cuda", enabled=(DEVICE == "cuda")):
            outputs = model(imgs)
            loss = criterion(outputs, labels) / GRAD_ACCUM_STEPS
        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        running_loss += loss.item() * GRAD_ACCUM_STEPS
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return running_loss / len(loader), macro_f1


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels, all_paths, all_probs = [], [], [], []
    for imgs, labels, paths in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast("cuda", enabled=(DEVICE == "cuda")):
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        running_loss += loss.item()

        probs = F.softmax(outputs.float(), dim=1)
        preds = probs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_paths.extend(paths)
        all_probs.extend(probs.cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return running_loss / len(loader), macro_f1, np.array(all_labels), np.array(all_preds), all_paths, np.array(all_probs)

## 6. Loop Utama — 6 Model x 3 Fold, dengan Try/Except per Model

Kalau 1 model gagal (misal nama checkpoint salah), dicatat di `failed_models` dan notebook lanjut ke model berikutnya — tidak berhenti total.

In [7]:
all_fold_results = []
detail_records = []
eval_counter = defaultdict(int)
failed_models = []

for model_cfg in MODEL_LIST:
    model_key = model_cfg["key"]
    img_size = model_cfg["img_size"]
    print(f"\n{'#'*70}\n### MODEL: {model_key}\n{'#'*70}")

    try:
        sss = StratifiedShuffleSplit(n_splits=N_FOLDS, test_size=FOLD_VAL_FRAC, random_state=SEED)

        for fold, (train_idx, val_idx) in enumerate(sss.split(full_df, full_df["label"]), start=1):
            print(f"\n{'='*64}\n---> [{model_key}] FOLD {fold}/{N_FOLDS} <---\n{'='*64}")
            train_df = full_df.iloc[train_idx]
            val_df = full_df.iloc[val_idx]

            weights = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=train_df["label"].values)
            class_weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)
            print(f"  Class weights: {dict(zip(CLASS_NAMES, weights.round(3)))}")

            model, mean, std = build_model(model_cfg, NUM_CLASSES)
            if fold == 1:
                n_total = sum(p.numel() for p in model.parameters())
                n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
                print(f"  Total params: {n_total:,} | Trainable (head): {n_train:,}")

            train_tf = build_transforms(mean, std, img_size, train=True)
            val_tf = build_transforms(mean, std, img_size, train=False)

            train_loader = DataLoader(TrashDataset(train_df, train_tf),
                                      batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True,
                                      persistent_workers=(NUM_WORKERS > 0), prefetch_factor=4 if NUM_WORKERS > 0 else None)
            val_loader = DataLoader(TrashDataset(val_df, val_tf),
                                    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
                                    persistent_workers=(NUM_WORKERS > 0), prefetch_factor=4 if NUM_WORKERS > 0 else None)

            criterion = nn.CrossEntropyLoss(weight=class_weights)
            optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                          lr=LR, weight_decay=WEIGHT_DECAY)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

            best_f1 = 0.0
            epochs_no_improve = 0
            best_path = MODELS_DIR / f"{model_key}_fold{fold}.pth"

            for epoch in range(1, NUM_EPOCHS + 1):
                tr_loss, tr_f1 = train_one_epoch(model, train_loader, criterion, optimizer)
                val_loss, val_f1, _, _, _, _ = evaluate(model, val_loader, criterion)
                scheduler.step(val_f1)

                marker = ""
                if val_f1 > best_f1:
                    best_f1 = val_f1
                    epochs_no_improve = 0
                    torch.save(model.state_dict(), best_path)
                    marker = " [*] best"
                else:
                    epochs_no_improve += 1

                print(f"  Epoch {epoch:2d}/{NUM_EPOCHS} | train_loss {tr_loss:.4f} F1 {tr_f1:.4f} | "
                      f"val_loss {val_loss:.4f} F1 {val_f1:.4f}{marker}")

                if epochs_no_improve >= PATIENCE:
                    print(f"  Early stopping.")
                    break

            model.load_state_dict(torch.load(best_path, weights_only=True))
            _, final_f1, y_true, y_pred, paths, probs = evaluate(model, val_loader, criterion)
            print(f"  >> [{model_key}] Fold {fold} best macro F1: {best_f1:.4f} | re-eval F1: {final_f1:.4f}")

            all_fold_results.append({"model": model_key, "fold": fold, "best_macro_f1": best_f1})

            for i, path in enumerate(paths):
                eval_counter[(model_key, path)] += 1
                if y_true[i] != y_pred[i]:
                    p = probs[i]
                    detail_records.append({
                        "model": model_key, "fold": fold, "filepath": path,
                        "true_label": CLASS_NAMES[y_true[i]],
                        "predicted_label": CLASS_NAMES[y_pred[i]],
                        "confidence_pct": round(float(p[y_pred[i]]) * 100, 2),
                        "true_label_confidence_pct": round(float(p[y_true[i]]) * 100, 2),
                        "prob_recyclable_pct": round(float(p[0]) * 100, 2),
                        "prob_electronic_pct": round(float(p[1]) * 100, 2),
                        "prob_organic_pct": round(float(p[2]) * 100, 2),
                    })

            del model, train_loader, val_loader, optimizer, scheduler
            gc.collect()
            torch.cuda.empty_cache()

    except Exception as e:
        print(f"\n  [GAGAL] Model {model_key} error: {e}")
        print(f"  Melanjutkan ke model berikutnya...")
        failed_models.append({"model": model_key, "error": str(e)})
        torch.cuda.empty_cache()
        continue

print(f"\n\n{'='*70}\nSemua model selesai diproses. Model gagal: {len(failed_models)}/{len(MODEL_LIST)}\n{'='*70}")
if failed_models:
    for f in failed_models:
        print(f"  - {f['model']}: {f['error']}")


######################################################################
### MODEL: deit_base
######################################################################

---> [deit_base] FOLD 1/3 <---
  Class weights: {'Recyclable': np.float64(0.884), 'Electronic': np.float64(2.232), 'Organic': np.float64(0.704)}


  Total params: 85,800,963 | Trainable (head): 2,307
  Epoch  1/15 | train_loss 0.2643 F1 0.9044 | val_loss 0.1564 F1 0.9424 [*] best
  Epoch  2/15 | train_loss 0.1682 F1 0.9362 | val_loss 0.1413 F1 0.9484 [*] best
  Epoch  3/15 | train_loss 0.1644 F1 0.9388 | val_loss 0.1351 F1 0.9488 [*] best
  Epoch  4/15 | train_loss 0.1462 F1 0.9424 | val_loss 0.1323 F1 0.9542 [*] best
  Epoch  5/15 | train_loss 0.1424 F1 0.9459 | val_loss 0.1319 F1 0.9520
  Epoch  6/15 | train_loss 0.1427 F1 0.9447 | val_loss 0.1328 F1 0.9514


KeyboardInterrupt: 

## 7. Ringkasan Hasil — Perbandingan Semua Model

In [ ]:
results_df = pd.DataFrame(all_fold_results)
print("Ringkasan 3-Fold CV per model (macro F1):\n")
for model_key in results_df["model"].unique():
    sub = results_df[results_df["model"] == model_key]
    f1s = sub["best_macro_f1"].values
    print(f"  {model_key}:")
    for _, row in sub.iterrows():
        print(f"    Fold {int(row['fold'])}: {row['best_macro_f1']:.4f}")
    print(f"    Rata-rata: {np.mean(f1s):.4f} (+/- {np.std(f1s):.4f})\n")

print("Pembanding model kamu sebelumnya:")
print("  ViT-B/16 SWAG        : 0.9636")
print("  EVA-02 Base          : 0.9646")
print("  BEiT-Base            : 0.9695")
print("  ConvNeXt V2-Large    : 0.9720")
print("  ViT-L/16 SWAG        : 0.9770")
print("  SwinV2-Base          : 0.9776  <- terbaik sejauh ini")

results_df.to_csv(OUTPUT_DIR / "six_new_models_fold_results.csv", index=False)
if failed_models:
    pd.DataFrame(failed_models).to_csv(OUTPUT_DIR / "six_new_models_failed.csv", index=False)

## 8. Laporan Misclassified (`.xlsx`, 2 sheet, kolom `model`)

In [ ]:
detail_df = pd.DataFrame(detail_records, columns=[
    "model", "fold", "filepath", "true_label", "predicted_label",
    "confidence_pct", "true_label_confidence_pct",
    "prob_recyclable_pct", "prob_electronic_pct", "prob_organic_pct",
])

summary_rows = []
if len(detail_df) > 0:
    for (model_key, path), g in detail_df.groupby(["model", "filepath"]):
        folds_wrong = sorted(g["fold"].unique().tolist())
        total_eval = eval_counter[(model_key, path)]
        summary_rows.append({
            "model": model_key,
            "filepath": path,
            "true_label": g["true_label"].iloc[0],
            "folds_wrong": ",".join(map(str, folds_wrong)),
            "total_wrong": len(g),
            "total_evaluated": total_eval,
            "error_rate": round(len(g) / total_eval, 3) if total_eval else None,
            "avg_confidence_pct": round(g["confidence_pct"].mean(), 2),
            "avg_true_label_confidence_pct": round(g["true_label_confidence_pct"].mean(), 2),
            "avg_prob_recyclable_pct": round(g["prob_recyclable_pct"].mean(), 2),
            "avg_prob_electronic_pct": round(g["prob_electronic_pct"].mean(), 2),
            "avg_prob_organic_pct": round(g["prob_organic_pct"].mean(), 2),
        })

summary_df = pd.DataFrame(summary_rows, columns=[
    "model", "filepath", "true_label", "folds_wrong", "total_wrong", "total_evaluated", "error_rate",
    "avg_confidence_pct", "avg_true_label_confidence_pct",
    "avg_prob_recyclable_pct", "avg_prob_electronic_pct", "avg_prob_organic_pct",
])
if len(summary_df) > 0:
    summary_df = summary_df.sort_values(["error_rate", "total_wrong"], ascending=False).reset_index(drop=True)

report_path = OUTPUT_DIR / "misclassified_report_six_new_models.xlsx"
with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
    detail_df.to_excel(writer, sheet_name="Detail", index=False)
    summary_df.to_excel(writer, sheet_name="Summary", index=False)

print(f"[SAVED] {report_path}")
print(f"  Detail  : {len(detail_df)} baris")
print(f"  Summary : {len(summary_df)} gambar unik")

## Catatan
- Kalau salah satu model OOM: turunkan `BATCH_SIZE` global, atau edit `MODEL_LIST` untuk jalankan subset saja (hapus entry yang bermasalah, run ulang khusus model itu dengan batch lebih kecil).
- Kalau `densenet264d` atau `coatnet_2_rw_224.sw_in12k` gagal load (dicatat di `failed_models`), berarti nama checkpoint pretrained perlu dicek ulang -- bisa saya bantu cari nama alternatif yang benar.
- Total waktu training 6 model x 3 fold ini kemungkinan panjang (bisa 6-12+ jam tergantung kecepatan tiap model). Pertimbangkan jalankan semalaman atau saat komputer kampus tidak dipakai.